## 5. 音频处理
转换器模型不仅可以处理文本，在处理语音方面也取得了巨大突破。语音与文本一样，都是人类语言的载体，也就都存在上下文依赖问题。所以只要能够让转换器模型读入语音，模型中的自注意力机制就能够发挥作用。它同样有助于更精准的表述每个语音在整段音频中的含义，进而提升模型识别语音含义的整体准确率。当然，音频并不仅限于语音，但本章讨论的主要还是语音。

### 5.1  音频分类与语音识别
转换器模型在音频处理方面的典型应用主要是音频分类（Audio Classification）和自动语音识别（Automatic Speech Recognition，ASR）。

#### 5.1.1  音频分类
使用transformers库执行音频分类任务非常简单，与上一章中自然语言处理任务几乎没有区别。音频分类在transformers中的任务名称为audio-classification，所以它的实现代码如下：

In [ ]:
# 可能需要安装FFmpeg
from transformers import pipeline

classifier = pipeline(task="audio-classification")
preds = classifier("/path/to/your/audio")
print(preds)

音频分类的默认模型为superb/wav2vec2-base-superb-ks，这是一个基于Wav2Vec 2.0微调后的语音模型。Wav2Vec 2.0类似于语言模型中的BERT模型，也是一种仅编码器形态的转换器模型。该模型预训练采用了16kHz的音频数据，所以执行上述代码要保证音频采样率为16kHz。如果音频采样率不是16kHz，音频分类的流水线会尝试使用FFmpeg（Fast Forward Moving Picture Experts Group）将其转换为16kHz，所以执行上述代码可能需要预先安装FFmpeg，可到https://www.ffmpeg.org/download.html下载该软件。名称中的superb-ks代表的是SUPERB（Speech processing Universal PERformance Benchmark）基准测试的关键字检测（Keyword Spotting，KS），包括yes、no、up、down、left、right、on、off、stop、go等。所以使用默认模型只能分类到以上几个类别中，一般可用于智能设备的语音控制。

#### 5.1.2  基于Wav2Vec识别语音
transformers库中的自动语音识别任务名称为automatic-speech-recognition：

In [ ]:
# 可能需要安装FFmpeg
from transformers import pipeline

classifier = pipeline(task="automatic-speech-recognition")
preds = classifier("/path/to/your/audio")
print(preds)

transformers自动语音识别的默认模型为facebook/wav2vec2-base-960h，这依然是基于Wav2Vec 2.0的语音模型。名称中的960h代表的是960小时的纯英文语音数据集，所以它也就只能识别英文语音。中文也有类似的语音数据集，比较著名的包括AISHELL系列、THCHS-30等等。Wav2Vec 2.0是一个仅编码器的转换器模型，它的直接输出并非文本，而是一组带有丰富语义的向量数据，需要通过CTC算法将它们转换为文本，详见书中讲解。

#### 5.1.3  基于Whisper识别语音
Wav2Vec 2.0这样的模型是一个“仅声学”模型，它会倾向于以表音的方式识别文本。这导致的一个直接后果就是生成的文本中容易出现拼写错误，并且生成的结果中一般不会有大小写字母的区分和标点符号（因为它们在音频中就不会出现）。执行下面的代码将出现不少语音识别错误：

In [ ]:
from transformers import pipeline
from datasets import load_dataset
from datasets import Audio

# 加载minds14数据集
minds = load_dataset("PolyAI/minds14", name="zh-CN", split="train")
minds = minds.cast_column("audio", Audio(sampling_rate=16000))
# 加载ASR流水线
model = "jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn"
# model = "jonatasgrosman/whisper-large-zh-cv11"
asr = pipeline("automatic-speech-recognition", model=model)
# 如果minds14下载不了，请使用本地文件
# example = "res/input5.3.wav"
example = minds[0]["audio"]["array"]
o = asr(example)

print(o["text"], "\n", example["transcription"])

原因是Wav2Vec 2.0更倾向于表音，而并不关心生成结果在语义上的正确性。这种状况直到2022年OpenAI发布了Whisper模型后才得到有效改善。Whisper模型采用了编码器-解码器形态，解码器的加入让Whisper在表音的基础上，更加注重生成结果的文本含义。可将模型替换为jonatasgrosman/whisper-large-zh-cv11，再次执行后可得到正确的结果。

#### 5.1.4  文本到语音
文本到语音（Text To Speech，TTS）与自动语音识别正好相反，是将文本转换为语音的任务。文本到语音任务在transformers中的名称为text-to-audio，使用该名称加载流水线即可将文本轻松转换为音频：

In [ ]:
from transformers import pipeline
from scipy.io.wavfile import write
import numpy as np

# 创建文本到音频流水线，并将文本转换为音频
synthesizer = pipeline(task="text-to-audio")
audio = synthesizer("I love this book. [laughs]")
print(audio)
# 提取音频波形数据和采样率
audio_array = audio["audio"][0]
sampling_rate = audio["sampling_rate"]

# 保存为 .wav 文件
output_file = "output5.4.wav"
write(output_file, sampling_rate, (audio_array * 32767).astype(np.int16))
print(f"Audio saved to {output_file}")

文本到语音任务还有一个别名text-to-speech，使用这个名称也可以加载音频模型做转换。文本到语音的默认模型为suno/bark-small，不仅可以生成语音、音乐和背景音，还能够模拟大笑、叹息和哭泣等声音。

### 5.2  音频数字化特征*
本节主要介绍了音频的一些数字化特征，包括频率、振幅、采样率、位深度、频谱、波形、时域和频域等。

#### 5.2.1  从声音到音频
由于物体振动是声波的来源，所以物体振动的频率和幅度决定了声音最基本的特征。物体振动的频率决定了声音的音高和音色，而物体振动的幅度则决定了声音的强度。频率的单位是赫兹，而振幅（或强度）的单位是分贝。声音转换为数字音频是通过模数转换器的采样过程来实现的，这其中就有了采样频率的问题。根据奈奎斯特极限（Nyquist Limit）定理的要求，在无噪声的理想信道中，采样率至少得是最高频率的两倍才有可能重构原始信号。人类听力敏感范围是1000~8000Hz，这就是为什么音频一般采用16KHz采样率的原因。

#### 5.2.2  波形数据
波形数据实际上就是音频文件最原始的采样点数据，将一个未经压缩的音频文件直接读取进来，得到的就是该音频的波形数据。Python中有许多库都可以做到这一点，下面的代码就是使用librosa库提取波形的示例：

In [ ]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
import numpy as np
import simpleaudio as sa
# 加载小号音频样例
array, sampling_rate = librosa.load(librosa.ex("trumpet"))
print(len(array), sampling_rate)
# 播放音频
play_obj = sa.play_buffer((array * 32767).astype(np.int16), 1, 2, sampling_rate)
# 绘制波形图
plt.figure().set_figwidth(10)
librosa.display.waveshow(array, sr=sampling_rate)
plt.show()

执行上面的代码可绘制如下图形：

![音频波形图](./images/wave.png)

#### 5.2.3  时频谱数据
频谱是音频在某一时间点上的频率强度分布，时频谱则将各个时间点上频谱串连起来，反映了频谱在时间维度上的变化过程。从数据结构上来说，频谱是一个只包含有频率及其强度的二维数组，而时频谱则是包含了时间、频率和强度的三维数组。频谱是从原始音频的振幅强度转化而来，从数学上说就是对波形数据做离散傅里叶变换（Discrete Fourier Transform，DFT）。使用Python的librosa库，可以轻松实现频谱的计算：

In [ ]:
import numpy as np
import librosa
import matplotlib.pyplot as plt

array, sampling_rate = librosa.load(librosa.ex("trumpet"))
# 计算短时傅里叶变换并转换为分贝
stft = librosa.stft(array)
stft_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)
print(array.shape, stft.shape)
plt.figure().set_figwidth(12)
librosa.display.specshow(stft_db, x_axis="time", y_axis="hz")
plt.colorbar()
plt.show()

执行上述代码，可绘制时频谱图如下：

![时频谱图](./images/time-fre.png)

#### 5.2.4  梅尔时频谱
人耳对不同频率在变化上的感知能力存在差异，对低频范围内的频率变化感知能力更强，而对高频范围内的频率变化则比较弱。经统计发现它们之间呈现大体的对数关系，这就是音频领域中著名的梅尔频率尺度（Mel Frequency Scale）。梅尔时频谱（Mel Frequency Spectrum）正是基于这一发现，以滤波器的形式过滤掉高频变化，从而在保留有价值音频特征的前提下压缩了数据大小。使用librosa库可轻松实现从波形数据到梅尔时频谱的转化：

In [ ]:
import numpy as np
import librosa
import matplotlib.pyplot as plt

array, sampling_rate = librosa.load(librosa.ex("trumpet"))
# 计算梅尔时频谱并转换为分贝
mel = librosa.feature.melspectrogram(y=array,sr=sampling_rate,
n_mels=80,fmax=8000)
print(array.shape, mel.shape)
mel_db = librosa.power_to_db(mel, ref=np.max)
plt.figure().set_figwidth(10)
librosa.display.specshow(mel_db,x_axis="time",y_axis="mel",
sr=sampling_rate,fmax=8000)
plt.colorbar()
plt.show()

执行上述代码，绘制如下图：

![梅尔时频谱图](./images/mel-time-fre.png)

### 5.3  特征提取与音频流水线
本节主要解析了transformers库在处理流程音频上的底层实现，虽然流水线在音频处理上还是由预处理、前向传播和后处理三个部分组成，但预处理组件由分词器变成了特征提取器（Feature Extractor）。本节还讲解了Wav2Vec2模型在音频处理上的实现，并详细介绍了Wav2Vec2模型中的特征提取器，具体请参考书中5.3节。

#### 5.3.1  特征提取器
特征提取器的主要职责包括重采样、归一化、填充和裁剪等，对于Whisper模型来说，特征提取器主要功能是把音频数据转换成梅尔时频谱图。具体请参考书中介绍。

#### 5.3.2  音频模型*
这一小节主要介绍了Wav2Vec2模型在音频处理上的实现，核心是Wav2Vec2中的卷积神经网络，具体请参考书中5.3.2节。

#### 5.3.3  多模态处理器及流水线拆解
这一小节介绍了多模态处理器及流水线拆解，具体请参考书中5.3.3节。

### 5.4  本章小结
本章介绍了转换器在音频处理中的应用，并在讲解音频处理知识的基础之上，对不同模型处理音频的底层原理做了详细介绍。
本章介绍了音频分类、自动语音识别和文语转换等三种音频处理任务，其中音频分类为典型的判别式任务，而自动语音识别和文语转换则为生成式任务。涉及到的音频模型包括Wav2Vec 2.0、Whisper和BARK等三种，其中Wav2Vec 2.0是仅编码器模型，适合处理基于对音频内容理解的音频分类任务；Whisper是编码器-解码器形态的模型，它在理解音频内容的基础上还能生成精准的文本内容；BARK则是仅解码器形态的模型，专门应用于音频生成类任务。在音频的预处理上，Wav2Vec 2.0直接以波形数据为输入，但会使用CNN对波形数据做特征提取；而Whisper在音频预处理上则采用了梅尔时频谱格式，所以Whisper模型就不会再使用CNN做二次处理了。
本章还简要介绍了音频相关的一些技术知识，它们是深入理解音频处理任务的必要基础。这包括声音的频率、振幅等基本特征，音频的位深度和采样率等概念，以及音频数据的波形、时频谱等多种表现形式等等。如果想在音频处理领域有所建树，需要对以上基础知识有更深入的研究。